# Qwen3.8-27B sıfır bütçeli Colab probe

Bu defter benchmark sonucu vaat etmez. Varsayılan yol büyük model indirmeden küçük bir surrogate ile ölçüm/checkpoint hattını doğrular. Q4 referansı ancak kullanıcı açıkça isterse, ücretsiz oturum en az 15 GiB VRAM gösterirse, checksum doğrulanırsa ve kullanıcı sabitlenmiş bir `llama-server` yolu verirse çalışabilir. Vision, MTP ve 262K context kapsam dışıdır.

In [ ]:
# Güvenlik ve maliyet politikası: bu değerleri gevşetmeyin.
import os
ZERO_SPEND = True
ALLOW_PAID_API = False
ALLOW_REMOTE_INFERENCE_API = False
USE_GOOGLE_DRIVE = False       # İsteğe bağlı kalıcı checkpoint
REQUEST_Q4_REFERENCE = False   # Varsayılan: büyük indirme yok
LLAMA_SERVER_PATH = ""        # Kullanıcı tarafından sabitlenmiş b10549 binary yolu
LLAMA_PERPLEXITY_PATH = ""    # İsteğe bağlı, aynı b10549 build'i
LLAMA_BUILD_MANIFEST_PATH = "" # qwen-lab-build.json; full commit kanıtı
LLAMA_EXPECTED_REVISION = "b10549"
LLAMA_EXPECTED_COMMIT = "b2e5e9b28b2484fbf94b543432ece638996a8b97"
assert ZERO_SPEND and not ALLOW_PAID_API and not ALLOW_REMOTE_INFERENCE_API
for key in ("OPENAI_API_KEY", "ANTHROPIC_API_KEY", "DASHSCOPE_API_KEY", "GOOGLE_API_KEY"):
    os.environ.pop(key, None)
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
print("Politika: sıfır harcama; ücretli/uzak inference API kapalı.")

In [ ]:
from pathlib import Path
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    LAB_ROOT = Path("/content/drive/MyDrive/qwen38-27b-4gb-lab")
else:
    LAB_ROOT = Path("/content/qwen38-27b-4gb-lab")
for name in ("models", "cache", "checkpoints", "results"):
    (LAB_ROOT / name).mkdir(parents=True, exist_ok=True)
STATE_PATH = LAB_ROOT / "checkpoints/run-state.json"
RESULT_PATH = LAB_ROOT / "results/latest-summary.json"
print({"lab_root": str(LAB_ROOT), "persistent": USE_GOOGLE_DRIVE})

In [ ]:
import json, platform, shutil, subprocess, time
def run_probe(command):
    try:
        return subprocess.run(command, capture_output=True, text=True, timeout=15, check=False).stdout.strip()
    except (FileNotFoundError, subprocess.TimeoutExpired):
        return None

gpu_csv = run_probe(["nvidia-smi", "--query-gpu=name,memory.total,memory.free,compute_cap,driver_version", "--format=csv,noheader,nounits"])
vram_bytes = 0
vram_free_bytes = 0
if gpu_csv:
    try:
        vram_bytes = int(float(gpu_csv.splitlines()[0].split(",")[1].strip()) * 1024**2)
        vram_free_bytes = int(float(gpu_csv.splitlines()[0].split(",")[2].strip()) * 1024**2)
    except (ValueError, IndexError):
        pass
meminfo = {}
if Path("/proc/meminfo").exists():
    for line in Path("/proc/meminfo").read_text(encoding="ascii").splitlines():
        key, value = line.split(":", 1)
        meminfo[key] = int(value.split()[0]) * 1024
resources = {
    "platform": platform.platform(),
    "python": platform.python_version(),
    "gpu_probe": gpu_csv,
    "vram_bytes": vram_bytes,
    "vram_free_bytes": vram_free_bytes,
    "disk_free_bytes": shutil.disk_usage(LAB_ROOT).free,
    "local_disk_free_bytes": shutil.disk_usage("/content").free,
    "ram_bytes": (os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES")) if hasattr(os, "sysconf") else None,
    "available_ram_bytes": meminfo.get("MemAvailable")
}
print(json.dumps(resources, indent=2, ensure_ascii=False))

In [ ]:
# Atomik checkpoint: runtime koparsa tamamlanan aşamalar yeniden çalışmaz.
def load_state():
    if STATE_PATH.exists():
        return json.loads(STATE_PATH.read_text(encoding="utf-8"))
    return {"schema_version": 1, "completed_stages": [], "events": []}

def save_state(state):
    temp = STATE_PATH.with_suffix(".tmp")
    temp.write_text(json.dumps(state, indent=2, ensure_ascii=False), encoding="utf-8")
    os.replace(temp, STATE_PATH)

def complete(stage, payload):
    if stage not in state["completed_stages"]:
        state["completed_stages"].append(stage)
    state["events"].append({"stage": stage, "time_unix": time.time(), "payload": payload})
    save_state(state)

state = load_state()
complete("resource_probe", resources)
print({"resumed_stages": state["completed_stages"]})

In [ ]:
VRAM_GATE_BYTES = 15 * 1024**3
Q4_SIZE_BYTES = 18973870432
Q4_RUNTIME_HEADROOM_BYTES = 3 * 1024**3
q4_memory_headroom = vram_free_bytes + (resources["available_ram_bytes"] or 0)
q4_gate = vram_bytes >= VRAM_GATE_BYTES and q4_memory_headroom >= Q4_SIZE_BYTES + Q4_RUNTIME_HEADROOM_BYTES
MODE = "q4_reference" if REQUEST_Q4_REFERENCE and q4_gate else "surrogate"
if REQUEST_Q4_REFERENCE and not q4_gate:
    print("Q4 isteği reddedildi: 15 GiB VRAM veya birleşik boş RAM/VRAM headroom kapısı geçmedi; surrogate zorlandı.")
if MODE == "q4_reference" and resources["disk_free_bytes"] < Q4_SIZE_BYTES + 2 * 1024**3:
    raise RuntimeError("Q4 için doğrulanmış dosya + 2 GiB boş disk yok.")
if MODE == "q4_reference" and resources["local_disk_free_bytes"] < 2 * 1024**3:
    raise RuntimeError("Colab geçici alanında indirme/runtime için 2 GiB headroom yok.")
complete("mode_selection", {"mode": MODE, "q4_resource_gate_passed": q4_gate, "combined_free_memory_bytes": q4_memory_headroom})
print({"mode": MODE, "q4_resource_gate_passed": q4_gate, "combined_free_memory_bytes": q4_memory_headroom})

## Güvenli varsayılan: küçük katman surrogate
Bu deney yalnız notebook kayıt/ölçüm yolunu sınar; Qwen kalite veya hız benchmark'ı değildir.

In [ ]:
if MODE == "surrogate" and "surrogate" not in state["completed_stages"]:
    import torch
    torch.manual_seed(38)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    layers = []
    for _ in range(4):
        layers.extend([torch.nn.Linear(512, 512, bias=False), torch.nn.SiLU()])
    toy = torch.nn.Sequential(*layers).to(device).eval()
    x = torch.randn(1, 32, 512, device=device)
    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
    started = time.perf_counter()
    with torch.inference_mode():
        for _ in range(10):
            x = toy(x)
    if device == "cuda":
        torch.cuda.synchronize()
    summary = {
        "status": "surrogate_completed_not_qwen_benchmark",
        "device": device,
        "elapsed_seconds": time.perf_counter() - started,
        "peak_cuda_bytes": torch.cuda.max_memory_allocated() if device == "cuda" else 0
    }
    complete("surrogate", summary)
elif MODE == "surrogate":
    summary = next(e["payload"] for e in reversed(state["events"]) if e["stage"] == "surrogate")
print(summary if MODE == "surrogate" else "Surrogate atlandı; Q4 kapısı açık.")

## Q4 referansı (varsayılan kapalı)
Bu hücre uzaktan inference yapmaz; sabitlenmiş dosyayı anonim Hugging Face indirmesiyle yerel diske alır. İndirme `huggingface_hub` cache üzerinden devam eder. Çalıştırmak için ≥15 GiB VRAM, açık istek ve kullanıcı tarafından hazırlanmış/sabitlenmiş `llama-server` gerekir. Sunucu yalnız geçici bir loopback portuna bağlanır.

In [ ]:
if MODE == "q4_reference":
    import hashlib
    from huggingface_hub import hf_hub_download
    q4 = {
        "repo_id": "ggml-org/Qwen3.8-27B-GGUF",
        "filename": "Qwen3.8-27B-Q4_K_M.gguf",
        "revision": "97c30c65c8d9a3e73f9fdfb50f1d1a669e9a2827",
        "sha256": "31629f53165ab6a7dad8c9847dcfd1fdf55829dac1e6e748f4a68581b0033d34"
    }
    expected_model_path = LAB_ROOT / "models" / q4["filename"]
    if "q4_download_verified" in state["completed_stages"] and expected_model_path.is_file():
        model_path = expected_model_path
    else:
        model_path = Path(hf_hub_download(**{k: q4[k] for k in ("repo_id", "filename", "revision")}, local_dir=LAB_ROOT / "models", cache_dir=LAB_ROOT / "cache/huggingface"))
    digest = hashlib.sha256()
    with model_path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(16 * 1024**2), b""):
            digest.update(chunk)
    if digest.hexdigest() != q4["sha256"]:
        raise RuntimeError("Q4 SHA-256 uyuşmuyor; dosyayı çalıştırmayın.")
    if "q4_download_verified" not in state["completed_stages"]:
        complete("q4_download_verified", {"path": str(model_path), "sha256": digest.hexdigest()})
    if not LLAMA_SERVER_PATH or not Path(LLAMA_SERVER_PATH).is_file():
        complete("q4_runtime_blocked", {"reason": "Pinned LLAMA_SERVER_PATH not supplied"})
        raise RuntimeError("Checksum geçti; fakat sabitlenmiş llama-server verilmedi. Otomatik, hareketli runtime kurulmaz.")
    if not LLAMA_BUILD_MANIFEST_PATH or not Path(LLAMA_BUILD_MANIFEST_PATH).is_file():
        complete("q4_runtime_blocked", {"reason": "qwen-lab-build.json not supplied"})
        raise RuntimeError("Full commit attestasyonu için qwen-lab-build.json zorunludur.")
    build_manifest = json.loads(Path(LLAMA_BUILD_MANIFEST_PATH).read_text(encoding="utf-8"))
    if build_manifest.get("revision") != LLAMA_EXPECTED_COMMIT or build_manifest.get("build") != LLAMA_EXPECTED_REVISION:
        complete("q4_runtime_blocked", {"reason": "build manifest revision mismatch"})
        raise RuntimeError("Build manifest b10549/full llama.cpp commit'ini doğrulamıyor.")
    server_path = Path(LLAMA_SERVER_PATH).resolve()
    manifest_binaries = {Path(item["path"]).resolve(): item["sha256"] for item in build_manifest.get("binaries", [])}
    server_digest = hashlib.sha256(server_path.read_bytes()).hexdigest()
    if manifest_binaries.get(server_path) != server_digest:
        complete("q4_runtime_blocked", {"reason": "llama-server binary is absent from or differs from build manifest"})
        raise RuntimeError("llama-server binary hash'i attested build manifestiyle eşleşmiyor.")
    version_run = subprocess.run([str(server_path), "--version"], capture_output=True, text=True, timeout=30, check=False)
    version_output = (version_run.stdout + "\n" + version_run.stderr).strip()
    expected_build = LLAMA_EXPECTED_REVISION.lstrip("b")
    if expected_build not in version_output or LLAMA_EXPECTED_COMMIT[:7] not in version_output:
        complete("q4_runtime_blocked", {"reason": "llama-server revision attestation failed", "version": version_output[-1000:]})
        raise RuntimeError("llama-server b10549/full SHA attestation başarısız.")
    complete("q4_runtime_attested", {"expected_build": LLAMA_EXPECTED_REVISION, "expected_commit": LLAMA_EXPECTED_COMMIT, "version": version_output[-1000:]})
    if "q4_reference_attempt" not in state["completed_stages"]:
        import socket, urllib.error, urllib.request
        with socket.socket() as port_probe:
            port_probe.bind(("127.0.0.1", 0))
            q4_port = port_probe.getsockname()[1]
        command = [str(server_path), "--model", str(model_path), "--offline", "--no-mmproj", "--host", "127.0.0.1", "--port", str(q4_port), "--ctx-size", "2048", "--parallel", "1", "--ubatch-size", "64", "--cache-type-k", "q8_0", "--cache-type-v", "q8_0", "--n-gpu-layers", "auto", "--fit", "on", "--fit-target", "1024", "--no-webui", "--no-cache-prompt", "--cache-ram", "0", "--no-cache-idle-slots", "--cache-reuse", "0"]
        log_path = LAB_ROOT / "checkpoints/q4-reference-server.log"
        opener = urllib.request.build_opener(urllib.request.ProxyHandler({}))
        started = time.monotonic()
        with log_path.open("w", encoding="utf-8") as server_log:
            server = subprocess.Popen(command, stdout=server_log, stderr=subprocess.STDOUT, text=True)
            try:
                deadline = time.monotonic() + 900
                while True:
                    if server.poll() is not None:
                        raise RuntimeError(f"llama-server model yüklerken kapandı: {server.returncode}")
                    try:
                        with opener.open(f"http://127.0.0.1:{q4_port}/health", timeout=5) as response:
                            health = json.loads(response.read())
                            if response.status == 200 and health.get("status") == "ok":
                                break
                    except (OSError, urllib.error.HTTPError, json.JSONDecodeError):
                        pass
                    if time.monotonic() >= deadline:
                        raise RuntimeError("llama-server 900 saniyede hazır olmadı")
                    time.sleep(1)
                payload = {"prompt": "Bir cümlede sıfır bütçe politikasını açıkla.", "temperature": 0, "samplers": ["temperature"], "seed": 424242, "cache_prompt": False, "return_tokens": True, "n_predict": 32, "stream": False}
                request = urllib.request.Request(f"http://127.0.0.1:{q4_port}/completion", data=json.dumps(payload, ensure_ascii=False).encode("utf-8"), headers={"Content-Type": "application/json"}, method="POST")
                with opener.open(request, timeout=900) as response:
                    completion = json.loads(response.read())
                if not str(completion.get("content", "")).strip() or not completion.get("tokens"):
                    raise RuntimeError("Q4 /completion boş içerik veya token listesi döndürdü")
                q4_summary = {"status": "completed_reference_smoke_only", "elapsed_seconds": time.monotonic() - started, "response_chars": len(completion["content"]), "response_tokens": len(completion["tokens"]), "server_log": str(log_path)}
                complete("q4_reference_attempt", q4_summary)
            except Exception as exc:
                complete("q4_runtime_failed", {"reason": f"{type(exc).__name__}: {exc}", "server_log": str(log_path)})
                raise
            finally:
                if server.poll() is None:
                    server.terminate()
                    try:
                        server.wait(timeout=30)
                    except subprocess.TimeoutExpired:
                        server.kill()
                        server.wait(timeout=10)
    else:
        q4_summary = next(e["payload"] for e in reversed(state["events"]) if e["stage"] == "q4_reference_attempt")
    if "short_eval_attempt" not in state["completed_stages"] and LLAMA_PERPLEXITY_PATH and Path(LLAMA_PERPLEXITY_PATH).is_file():
        ppl_version = subprocess.run([LLAMA_PERPLEXITY_PATH, "--version"], capture_output=True, text=True, timeout=30, check=False)
        ppl_attestation = (ppl_version.stdout + "\n" + ppl_version.stderr).strip()
        if Path(LLAMA_PERPLEXITY_PATH).resolve().parent != server_path.parent:
            complete("short_eval_blocked", {"reason": "llama-perplexity is not from the attested build directory"})
        elif expected_build not in ppl_attestation or LLAMA_EXPECTED_COMMIT[:7] not in ppl_attestation:
            complete("short_eval_blocked", {"reason": "llama-perplexity revision attestation failed", "version": ppl_attestation[-1000:]})
        else:
            eval_path = LAB_ROOT / "checkpoints/short-public-domain-eval.txt"
            eval_path.write_text("The sun rises in the east. Water freezes at zero degrees Celsius. Ankara is the capital of Türkiye.", encoding="utf-8")
            ppl_command = [LLAMA_PERPLEXITY_PATH, "-m", str(model_path), "-f", str(eval_path), "-c", "512", "-b", "512", "--chunks", "1", "--n-gpu-layers", "auto", "--fit", "on", "--fit-target", "1024"]
            ppl_run = subprocess.run(ppl_command, capture_output=True, text=True, timeout=900, check=False)
            ppl_summary = {"returncode": ppl_run.returncode, "stdout_tail": ppl_run.stdout[-500:], "stderr_tail": ppl_run.stderr[-500:]}
            complete("short_eval_attempt" if ppl_run.returncode == 0 else "short_eval_failed", ppl_summary)
    elif "short_eval_attempt" not in state["completed_stages"]:
        complete("short_eval_blocked", {"reason": "Pinned LLAMA_PERPLEXITY_PATH not supplied; no automatic runtime install"})
    print(q4_summary)
else:
    print("Q4 indirilmedi/çalıştırılmadı; güvenli surrogate modu aktif.")

In [ ]:
# Küçük ve anonim özet; ham prompt/çıktı kaydetmez.
result = {
    "schema_version": 1,
    "claim": "Probe result only; a completed Q4 attempt is reference smoke, not a full Qwen benchmark.",
    "mode": MODE,
    "resources": resources,
    "completed_stages": state["completed_stages"]
}
temp_result = RESULT_PATH.with_suffix(".tmp")
temp_result.write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")
os.replace(temp_result, RESULT_PATH)
print({"result": str(RESULT_PATH), "state": str(STATE_PATH)})